# Planktonzilla — sampling locations on the world map

This notebook plots **where each sample of the Planktonzilla dataset was collected**, on a
world map, **colored by its source dataset**.

**Data source.** Coordinates live in the full consolidated dataset
[`project-oceania/planktonzilla-17M`](https://huggingface.co/datasets/project-oceania/planktonzilla-17M)
(columns `Latitude`, `Longitude`, `dataset`). The derived `planktonzilla_only_plankton`
dataset used elsewhere drops the geographic columns, so it cannot be used here.

**Geographic coverage caveat.** Only the sources whose images came through the EcoTaxa /
WHOI APIs carry per-sample coordinates:

| Has coordinates (appear on the map) | No coordinates (absent from the map) |
| --- | --- |
| `whoi`, `flowcamnet`, `uvp6net`, `zooscan`, `planktoscope`, `global_uvp5` | `isiisnet`, `lensless`, `medplanktonset`, `zoocamnet`, `planktonset1.0`, `syke_ifcb_2022` |

Samples from the no-coordinate sources have `Latitude`/`Longitude` = `NaN` and are
silently excluded from the map (their counts are reported in the coverage table below).

**Efficiency.** The full dataset embeds ~17M images. We read **only** the three metadata
columns via parquet column projection, so the image bytes are never downloaded.

> **Auth:** `planktonzilla-17M` may be private. Set `HF_TOKEN` in your environment or run
> `huggingface-cli login` before running the loading cell.


In [ ]:
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

# Full consolidated dataset — the only one that keeps per-sample coordinates.
REPO_ID = "project-oceania/planktonzilla-17M"
GEO_COLS = ["Latitude", "Longitude", "dataset"]

# Local cache so the remote metadata read happens only once.
CACHE_PATH = Path("planktonzilla_sample_locations.parquet")

# Shared map styling (matches the previous Plotly look).
LAND_COLOR = "#f3f3f3"
OCEAN_COLOR = "#deeef7"
COUNTRY_COLOR = "#d2d2d2"
COAST_COLOR = "#909090"

In [ ]:
import seaborn as sns

## 1. Load sample coordinates (metadata-only, no image download)

We list the dataset's parquet shards on the Hugging Face filesystem and read just the
`Latitude`, `Longitude`, `dataset` columns with pyarrow's column projection. The `image`
column is never touched, so this stays light even though the dataset has ~17M rows.

In [ ]:
def load_sample_geo(repo_id=REPO_ID, columns=GEO_COLS, cache_path=CACHE_PATH):
    """Read ONLY the geo + source columns from the dataset's parquet shards.

    Uses parquet column projection over the Hugging Face filesystem, so the (huge)
    ``image`` column is never downloaded. Requires HF auth if the dataset is private:
    set ``HF_TOKEN`` or run ``huggingface-cli login``.
    """
    if cache_path.exists():
        print(f"Loading cached metadata from {cache_path} …")
        return pd.read_parquet(cache_path)

    import pyarrow.dataset as pads
    from huggingface_hub import HfFileSystem
    from pyarrow.fs import FSSpecHandler, PyFileSystem

    fs = HfFileSystem()  # picks up HF_TOKEN from env / cached login automatically
    pattern = f"datasets/{repo_id}/**/*.parquet"
    parquet_files = fs.glob(pattern)
    if not parquet_files:
        raise FileNotFoundError(
            f"No parquet shards found at hf://{pattern}. Check the repo id and that "
            "you are authenticated for this (possibly private) dataset."
        )
    print(f"Found {len(parquet_files)} parquet shard(s); reading columns {columns} …")

    pa_fs = PyFileSystem(FSSpecHandler(fs))
    table = pads.dataset(parquet_files, filesystem=pa_fs, format="parquet")

    available = set(table.schema.names)
    use_cols = [c for c in columns if c in available]
    missing = set(columns) - available
    if missing:
        print(f"⚠️  Columns absent from the dataset and skipped: {sorted(missing)}")

    df = table.to_table(columns=use_cols).to_pandas()
    df.to_parquet(cache_path, index=False)
    print(f"Read {len(df):,} rows. Cached to {cache_path}.")
    return df


geo = load_sample_geo()
geo.head()

## 2. Coverage: how many samples are geolocated, per source

Sources without coordinates (all `NaN`) show `0` here and will not appear on the map.

In [ ]:
geo["has_coords"] = geo["Latitude"].notna() & geo["Longitude"].notna()

coverage = (
    geo.groupby("dataset", dropna=False)
    .agg(n_samples=("dataset", "size"), n_with_coords=("has_coords", "sum"))
    .assign(pct_with_coords=lambda d: (100 * d["n_with_coords"] / d["n_samples"]).round(1))
    .sort_values("n_samples", ascending=False)
)

total = len(geo)
total_geo = int(geo["has_coords"].sum())
print(f"Total samples:            {total:,}")
print(f"Samples with coordinates: {total_geo:,} ({100 * total_geo / total:.1f}%)")
print(f"Source datasets:          {geo['dataset'].nunique()}")
print()
coverage

## 3. Aggregate to unique sampling locations

Plotting millions of overlapping points is slow and unreadable. We round coordinates to a
small grid and count samples per `(dataset, location)`, then size each marker by its count.
Lower `ROUND_DECIMALS` to coarsen the grid (1 ≈ 11 km, 2 ≈ 1.1 km at the equator).

In [ ]:
ROUND_DECIMALS = 2  # grid resolution for merging nearby points

pts = geo.loc[geo["has_coords"], ["Latitude", "Longitude", "dataset"]].copy()

# Drop physically impossible coordinates.
valid = pts["Latitude"].between(-90, 90) & pts["Longitude"].between(-180, 180)
dropped = int((~valid).sum())
if dropped:
    print(f"Dropped {dropped:,} rows with out-of-range coordinates.")
pts = pts[valid]

pts["lat"] = pts["Latitude"].round(ROUND_DECIMALS)
pts["lon"] = pts["Longitude"].round(ROUND_DECIMALS)

locations = (
    pts.groupby(["dataset", "lat", "lon"], as_index=False)
    .size()
    .rename(columns={"size": "n_samples"})
    .sort_values("n_samples", ascending=False)
)

print(
    f"{len(pts):,} geolocated samples → {len(locations):,} unique map locations "
    f"(rounded to {ROUND_DECIMALS} decimals)."
)
locations.head()

## 4. World map — all sources together

Each marker is a sampling location, colored by source dataset and sized by the number of
samples collected there. The figure is rendered with matplotlib + cartopy and also saved to
a PNG image.

In [ ]:
# Stable color per source dataset (sorted so colors are reproducible across figures).
sources = sorted(locations["dataset"].unique())
cmap = plt.get_cmap("tab20" if len(sources) > 10 else "tab10")
colors = {src: cmap(i % cmap.N) for i, src in enumerate(sources)}

# Scale sample counts to a visible marker-area range (matplotlib `s` is area in points²).
S_MIN, S_MAX = 6.0, 600.0
counts = locations["n_samples"].to_numpy()
sizes = S_MIN + (S_MAX - S_MIN) * (counts / counts.max())

fig = plt.figure(figsize=(14, 8))
ax = plt.axes(projection=ccrs.Robinson())
ax.set_global()
ax.add_feature(cfeature.OCEAN, facecolor=OCEAN_COLOR)
ax.add_feature(cfeature.LAND, facecolor=LAND_COLOR)
ax.add_feature(cfeature.BORDERS, edgecolor=COUNTRY_COLOR, linewidth=0.4)
ax.add_feature(cfeature.COASTLINE, edgecolor=COAST_COLOR, linewidth=0.5)

for src in sources:
    m = (locations["dataset"] == src).to_numpy()
    ax.scatter(
        locations.loc[m, "lon"],
        locations.loc[m, "lat"],
        s=sizes[m],
        color=colors[src],
        alpha=0.65,
        edgecolors=(0.16, 0.16, 0.16, 0.5),
        linewidths=0.3,
        transform=ccrs.PlateCarree(),
    )

# Fixed-size legend handles (marker size on the map encodes sample count, not source).
handles = [
    Line2D([0], [0], marker="o", linestyle="", markersize=8, markerfacecolor=colors[s], markeredgecolor="none", label=s)
    for s in sources
]
ax.legend(handles=handles, title="Source dataset", loc="lower left", fontsize=8, framealpha=0.9)
ax.set_title("Planktonzilla sampling locations by source dataset")

out_png = Path("world_map_sample_locations.png")
fig.savefig(out_png, dpi=150, bbox_inches="tight")
print(f"Saved map to {out_png.resolve()}")
plt.show()

## 5. Per-source footprint (small multiples)

One mini-map per source dataset, to compare where each campaign sampled.

In [ ]:
# Reuse the same deterministic source->color mapping as the combined map.
sources = sorted(locations["dataset"].unique())
cmap = plt.get_cmap("Dark2" if len(sources) > 10 else "Dark2")
colors = {src: cmap(i % cmap.N) for i, src in enumerate(sources)}

FS_MIN, FS_MAX = 11.0, 140.0  # smaller marker range for the mini-maps

ncols = 2
nrows = int(np.ceil(len(sources) / ncols))
fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(5.5*ncols, 3 * nrows),
    subplot_kw={"projection": ccrs.PlateCarree()},
)
axes = np.atleast_1d(axes).ravel()

for ax, src in zip(axes, sources):
    ax.set_global()
    ax.add_feature(cfeature.OCEAN, facecolor=OCEAN_COLOR)
    ax.add_feature(cfeature.LAND, facecolor=LAND_COLOR)
    ax.add_feature(cfeature.COASTLINE, edgecolor=COAST_COLOR, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, edgecolor=COUNTRY_COLOR, linewidth=0.4)
    ax.gridlines( 
        draw_labels=False,
        linewidth=0.3,
        color="gray",
        alpha=0.5,
        linestyle="--",
    )

    sub = locations[locations["dataset"] == src]
    c = sub["n_samples"].to_numpy()
    s = FS_MIN + (FS_MAX - FS_MIN) * (c / c.max())
    ax.scatter(
        sub["lon"],
        sub["lat"],
        s=s,
        color=colors[src],
        alpha=0.83,
        edgecolors="k",
        lw=0.29,
        transform=ccrs.PlateCarree(),
    )
    ax.set_title(src.upper())

# Hide any unused subplot slots.
for ax in axes[len(sources):]:
    ax.axis("off")

# fig.suptitle("Sampling footprint per source dataset", fontsize=13)
fig.tight_layout()
plt.show()
fig.savefig("sampling_footprint_per_dataset.png", dpi=350, bbox_inches="tight")

## 6. Inferred dataset-level locations (for the non-geolocated sources)

The six "no-coordinate" sources flagged at the top of this notebook don't carry
per-sample GPS, so they never appear on the maps above. We researched each one's
collection site from its **source paper / data archive** and place a single
representative point per dataset here.

**Caveat — read before reusing these points.** These are dataset-level *inferred
centroids* or fixed anchor stations, **not** recovered per-sample coordinates. They are
drawn as hollow rings to stay visually distinct from the measured locations in section 4.
Two sources have no single plottable point and are intentionally omitted: `planktoscope`
(a diffuse, multi-campaign global reference set) and `lensless` (a purchased Carolina
Biological lab culture with no field provenance).

| Dataset | Inferred site | Lat, Lon | Confidence |
| --- | --- | --- | --- |
| `zoocamnet` | Bay of Biscay shelf — PELGAS, R/V *Thalassa* | 45.5, −2.5 | high |
| `isiisnet` | Ligurian Sea off Nice — VISUFRONT | 43.5, 7.55 | high |
| `medplanktonset` | Gulf of Naples — LTER-MareChiara (SZN) | 40.82, 14.25 | high |
| `syke_ifcb_2022` | Baltic Sea — Utö station + ferrybox transects | 59.78, 21.37 | high |
| `planktonset1.0` | Straits of Florida — R/V *F.G. Walton Smith*, ISIIS-2 | 25.15, −80.55 | high |
| `zoolake` | **Lake Greifensee**, Switzerland — Eawag DSPC | 47.35, 8.68 | high |
| `sykezooscan2024` | Baltic Sea — sub-basin unspecified (inferred) | ~59.5, 21.4 | low |
| `planktoscope` | *diffuse / global* multi-campaign reference set | — | n/a |
| `lensless` | *lab/cultured* — purchased culture, no field site | — | n/a |

Full coordinates, regions, confidence and source DOIs are saved alongside this notebook
in `inferred_dataset_locations.csv`.

In [ ]:
# ── Inferred dataset-level collection locations ──────────────────────────────
# The "no-coordinate" sources flagged at the top of this notebook ship NO per-sample
# GPS, so they never appear on the maps above. We researched each one's collection
# site from its source paper / data archive and place a single representative point
# PER DATASET below (an inferred centroid or a fixed anchor station) — NOT recovered
# per-sample coordinates. They are drawn as hollow rings to stay visually distinct
# from the measured locations in section 4.
inferred = pd.DataFrame(
    [
        # dataset,          lat,    lon,   region,                               confidence
        ("zoocamnet",       45.50,  -2.50, "Bay of Biscay shelf (PELGAS)",       "high"),
        ("isiisnet",        43.50,   7.55, "Ligurian Sea off Nice (VISUFRONT)",  "high"),
        ("medplanktonset",  40.82,  14.25, "Gulf of Naples (LTER-MareChiara)",   "high"),
        ("syke_ifcb_2022",  59.78,  21.37, "Baltic Sea (Uto station)",           "high"),
        ("planktonset1.0",  25.15, -80.55, "Straits of Florida",                 "high"),
        ("zoolake",         47.35,   8.68, "Lake Greifensee, CH",                "high"),
        ("sykezooscan2024", 59.50,  21.40, "Baltic Sea (inferred, N. Baltic)",   "low"),
    ],
    columns=["dataset", "lat", "lon", "region", "confidence"],
)
# Documented but location-less — intentionally NOT plotted as single points:
#   * planktoscope — diffuse, multi-campaign global reference set (no single site)
#   * lensless     — purchased lab culture (Carolina Biological); no field location

fig = plt.figure(figsize=(14, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_global()
ax.add_feature(cfeature.OCEAN, facecolor=OCEAN_COLOR)
ax.add_feature(cfeature.LAND, facecolor=LAND_COLOR)
ax.add_feature(cfeature.BORDERS, edgecolor=COUNTRY_COLOR, linewidth=0.4)
ax.add_feature(cfeature.COASTLINE, edgecolor=COAST_COLOR, linewidth=0.4)

edge_color = {"high": "#c0392b", "low": "#e67e22"}
marker_shape = {"high": "o", "low": "^"}
for _, r in inferred.iterrows():
    ax.scatter(
        r["lon"],
        r["lat"],
        s=110,
        facecolors="none",  # hollow ring = inferred, not measured
        edgecolors=edge_color[r["confidence"]],
        linewidths=1.8,
        marker=marker_shape[r["confidence"]],
        transform=ccrs.PlateCarree(),
        zorder=5,
    )
    ax.text(
        r["lon"] + 1.5,
        r["lat"] + 1.2,
        r["dataset"],
        transform=ccrs.PlateCarree(),
        fontsize=7,
        color="#333333",
        zorder=6,
    )

handles = [
    Line2D([0], [0], marker="o", linestyle="", markerfacecolor="none",
           markeredgecolor="#c0392b", markersize=9, label="inferred — high confidence"),
    Line2D([0], [0], marker="^", linestyle="", markerfacecolor="none",
           markeredgecolor="#e67e22", markersize=9, label="inferred — low confidence"),
]
ax.legend(handles=handles, title="Dataset-level inferred site (not per-sample GPS)",
          loc="lower left", fontsize=8, framealpha=0.9)
ax.set_title("Inferred collection locations for the non-geolocated source datasets")

out_png = Path("world_map_inferred_locations.png")
fig.savefig(out_png, dpi=150, bbox_inches="tight")
print(f"Saved inferred-location map to {out_png.resolve()}")
plt.show()

In [ ]:
co=coverage # .sort_index()

In [ ]:
total_samples = co["n_samples"].sum()

In [ ]:
co["pct"] = co["n_samples"] / total_samples * 100

In [ ]:
co.sort_values("pct", ascending=True, inplace=True)

In [ ]:
co

In [ ]:
def label_spread_heuristic(ys, threshold=0.15, multiplier=1.0):
    """Heuristic label spread that keeps the wedge order but allows uneven spacing.

    This is a simpler alternative to the uniform spread above. It tries to keep each
    label's y-position close to its wedge's natural angle, but if two labels collide,
    it nudges them apart until they no longer overlap. The result is a more compact
    layout that still avoids text collisions.
    """
    overlap = True
    ys = [y * multiplier for y in ys]
    while overlap:
        overlap = False
        for k in range(1, len(ys)):
            if np.abs(ys[k-1] - ys[k]) < threshold:  # threshold is a small buffer
                overlap = True
                ys[k-1] = ys[k-1] + threshold
                break
                # if ys[k] < y_lo:
                #     ys[k] = y_lo    
    return ys

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), subplot_kw=dict(aspect="equal"))

dataset_labels = [f"{label} — {co.loc[label, 'pct']:.2f}%" for label in co.index.to_list()]
data = co.n_samples.to_list()

colors = plt.get_cmap('PuBuGn')(np.linspace(0, 0.74, 6))

pie = ax.pie(data, wedgeprops=dict(width=0.47, ec="k"), startangle=160,colors=colors)

bbox_props = dict(boxstyle="round,pad=0.3", fc="w", ec="none", lw=0.74, alpha=1)
kw = dict(arrowprops=dict(arrowstyle="-"),
          bbox=bbox_props, zorder=0, va="center")

ys_right = [np.sin(np.deg2rad((p.theta2 - p.theta1)/2. + p.theta1)) for p in pie.wedges if np.cos(np.deg2rad((p.theta2 - p.theta1)/2. + p.theta1)) >= 0]
ys_left = [np.sin(np.deg2rad((p.theta2 - p.theta1)/2. + p.theta1)) for p in pie.wedges if np.cos(np.deg2rad((p.theta2 - p.theta1)/2. + p.theta1)) < 0]

ys_right_fix = label_spread_heuristic(ys_right, multiplier=1.28)
ys_left_fix = label_spread_heuristic(ys_left, multiplier=1.28)

labels_right = [dataset_labels[i] for i in range(len(pie.wedges)) if np.cos(np.deg2rad((pie.wedges[i].theta2 - pie.wedges[i].theta1)/2. + pie.wedges[i].theta1)) >= 0]
labels_left = [dataset_labels[i] for i in range(len(pie.wedges)) if np.cos(np.deg2rad((pie.wedges[i].theta2 - pie.wedges[i].theta1)/2. + pie.wedges[i].theta1)) < 0]
x_right = [np.cos(np.deg2rad((p.theta2 - p.theta1)/2. + p.theta1)) for p in pie.wedges if np.cos(np.deg2rad((p.theta2 - p.theta1)/2. + p.theta1)) >= 0]
x_left = [np.cos(np.deg2rad((p.theta2 - p.theta1)/2. + p.theta1)) for p in pie.wedges if np.cos(np.deg2rad((p.theta2 - p.theta1)/2. + p.theta1)) < 0] 

angles_right = [(p.theta2 - p.theta1)/2. + p.theta1 for p in pie.wedges if np.cos(np.deg2rad((p.theta2 - p.theta1)/2. + p.theta1)) >= 0]
angles_left = [(p.theta2 - p.theta1)/2. + p.theta1 for p in pie.wedges if np.cos(np.deg2rad((p.theta2 - p.theta1)/2. + p.theta1)) < 0]

wedges_right = [p for p in pie.wedges if np.cos(np.deg2rad((p.theta2 - p.theta1)/2. + p.theta1)) >= 0]
wedges_left = [p for p in pie.wedges if np.cos(np.deg2rad((p.theta2 - p.theta1)/2. + p.theta1)) < 0]
wedges = wedges_right + wedges_left

labels = labels_right + labels_left
xs = x_right + x_left
ys = ys_right + ys_left
ys_fix = ys_right_fix + ys_left_fix
angles = angles_right + angles_left


for i, (x, y, y_fix, angle, label) in enumerate(zip(xs, ys, ys_fix, angles, labels)):
    horizontalalignment = {-1: "right", 1: "left"}[int(np.sign(x))]
    connectionstyle = f"arc, rad=5, angleB={angle}, angleA={90 + 90*np.sign(x)}, armA=92, armB=29" # ,rad=5,angleA={0 if x >= 180 else 0},"
    kw["arrowprops"].update({"connectionstyle": connectionstyle}) 
    kw["bbox"].update({"fc": wedges[i].get_facecolor()})
    ax.annotate(labels[i], xy=(x, y), xytext=(1.73*np.sign(x), y_fix),
                horizontalalignment=horizontalalignment, **kw)

plt.show()
fig.savefig("datasets-pie.png", dpi=350)